# Try the research-assistant agent

Two ways to try this repo's agent from a notebook:

- **Option A - quick local invoke.** Builds the agent in-process with the
  OSS `deepagents` library, reusing this repo's `instructions.md` and
  `tools/search.py`. No `mda` CLI or server needed - fastest way to iterate.
- **Option B - talk to the real managed agent.** Connects to the agent
  exactly as `agent.py` defines it, served locally by `mda dev .`.

**Prerequisites**

```bash
uv sync --extra notebook
# fill OPENAI_API_KEY (and LANGSMITH_API_KEY for option B) in ../.env
jupyter notebook notebooks/try_agent.ipynb
```

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Notebook runs from notebooks/, but the agent's code (tools/, instructions.md,
# .env) lives one level up at the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

import os

assert os.environ.get("OPENAI_API_KEY"), (
    f"Set OPENAI_API_KEY in {REPO_ROOT / '.env'} first"
)

## Option A - quick local invoke (no `mda dev` needed)

In [2]:
from deepagents import create_deep_agent

from tools.search import internet_search

instructions = (REPO_ROOT / "instructions.md").read_text()

local_agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[internet_search],
    system_prompt=instructions,
)

result = local_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is LangChain Managed Deep Agents?"}
        ]
    }
)
print(result["messages"][-1].content)

[{'id': 'rs_03f178df14580117006a90edeaa5f087d0bed7cd0447c52fb7', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqkO3xpINWr1G-96H-9tH193zYB_RIOlyNw5C0kyIc6SX79tsA0YqsdTgay-UbzY83mKuRTWhiyaITiaiF4kJwFQimrxS6DAEvxMiQ8acljty3BlyZmrH_8twU5voU4sxzacwlR99nFdULFyN5Ot9Rh6G6ENg-7AO5bJqXdZSreZeAuQ_U3dgHBKPHbX-HSA61xU5liYscm2-lBFbHTNXFutwsmsZgcP4BCRHJZamu2LPw6K03qnLbpqH9jJb6g6Skk_JDcwwPNvRXwA4dszIl-7bel3KYKWJkGRg9uIRMUEibbEzFBon6Sv7rb7o464JEEisYaD4yz-cCvo1rlXXjdQorJUmnmxqNFRW1rGnjjB6mF9D-X4RqXT2OJBC7-AgQ4cAEiJ61KnUKnRipVKWuZpxeDqyRWYt9ZCTXJgyF7NOAkkezsczv5V52D6FWOCI7k6MoSiw8BXe16fckcKat5qpJPdg2jYUTGra1mxgecz9LxwUQqTTIPJvPi6IbqewC9TEIT96XUTqZVmDSZuPIkT6JDms2pRRQuLZq_l2YwTLJTC_SgaLILn7FLWhbZHAV7l0Qk0ApdUuS8hFBSb4FdMBeRqDC18p_gBRr5XoEvK-K1d4THzd1NlRMg2jm7wgPPG-Q16HjMTXbSQFZTiqJy76ecPlMxDfYzx-vpFC3m-fQCRgfqVsqw9vEOPjf1JTHzqgJmv2FRiAP1B07u04dU_SzsCGUAKHKfITJUFi2ojZoyMs2WRHt5fLTzqtX40Q-LISUNOkpZI9N4IEJ8gWEgi-ymi6Jnan9gG04B1m353oCHvw1Ku8pQ2jYeNOxMEBUnSCywA62ZxpmaY2blzWD09

## Option B - talk to the managed agent via `mda dev`

In a separate terminal, from the repo root, run:

```bash
mda dev .
```

It prints the local Agent Server URL (default `http://127.0.0.1:2024`) and
the graph/assistant name to use below - it's usually the agent `name` from
`agent.py` (`"research-assistant"`), but check the printed Studio URL /
server logs and adjust `ASSISTANT_ID` if it differs.

In [ ]:
from langgraph_sdk import get_sync_client

SERVER_URL = "http://127.0.0.1:2024"
ASSISTANT_ID = "research-assistant"

client = get_sync_client(url=SERVER_URL)

thread = client.threads.create()
for chunk in client.runs.stream(
    thread["thread_id"],
    ASSISTANT_ID,
    input={
        "messages": [
            {"role": "user", "content": "What is LangChain Managed Deep Agents?"}
        ]
    },
    stream_mode="values",
):
    if chunk.event == "values":
        print(chunk.data["messages"][-1]["content"])